# Experiment 3: Combining Temperature History and City Identity as Input Features

Companion notebook to [`README.md`](README.md) for the DAAN 570 course project (Temperature Prediction with Deep Learning).

**Hypothesis**: `exp01_temperature_feature` (the target's own recent history as an input) and `exp02_city_embedding` (a learned city identity embedding) each independently changed test MAE/RMSE versus the weather-only baseline. Are their effects additive when applied together, or do they interact?

**What changed vs. the baseline**: both prior changes are applied at once, unmodified from how each experiment introduced them -- see [`README.md`](README.md) for the full rationale. `FEATURE_COLUMNS` is exp01's list (baseline features plus `temperature_2m`); city identity is injected as a second model input (a learned `Embedding`) via `city_aware=True` in `src/models/transformer/training.py`'s `run_training()`, exp02's change.

This notebook can run either locally (if `data/splits/` is already on your machine) or on Google Colab with a GPU runtime. The first code cell below handles both cases automatically.


## Setup

**Running on Colab**: this cell clones the (private) repo, installs dependencies, and pulls the Git-LFS-tracked raw data. One-time setup: create a fine-grained, read-only GitHub PAT for this repo and store it in Colab's Secrets manager (key icon in the left sidebar) under the name `GITHUB_PAT`, then grant this notebook access to it when prompted. Also select **Runtime > Change runtime type > GPU** before running.

**Running locally**: this cell detects that it's not on Colab and does nothing -- it assumes you already have the repo cloned, dependencies installed (`pip install -r requirements.txt`), and `data/splits/` available (or `data/raw/` present, from which splits regenerate automatically).

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

BRANCH = "exp03_city_and_temperature_features"


def run(cmd):
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed (exit {result.returncode}): {cmd}")


try:
    import google.colab
    from google.colab import userdata

    REPO_DIR = Path("/content/repo")
    token = userdata.get("GITHUB_PAT")
    repo_url = f"https://{token}@github.com/LarryGreen-alt/Temperature_Prediction_DeepLearning.git"

    # A directory can be left behind by a previous failed attempt without
    # being a real git checkout -- only trust it if .git is actually there.
    if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir():
        shutil.rmtree(REPO_DIR)

    if (REPO_DIR / ".git").is_dir():
        run(f"git -C {REPO_DIR} fetch -q origin {BRANCH}")
        run(f"git -C {REPO_DIR} checkout -q {BRANCH}")
        run(f"git -C {REPO_DIR} pull -q origin {BRANCH}")
    else:
        run(f"git clone -q -b {BRANCH} {repo_url} {REPO_DIR}")

    get_ipython().run_line_magic("cd", str(REPO_DIR))
    run("pip install -q -r requirements.txt")
    run("apt-get -qq install -y git-lfs")
    # --force: the git-lfs apt package's post-install step already sets up
    # this same hook globally, so a plain `install` refuses as a safety
    # check against clobbering a *different*, unrelated pre-push hook.
    run("git lfs install --force")
    run("git lfs pull")

    PROJECT_ROOT = REPO_DIR
except ImportError:

    def find_project_root(marker="weather_main.py"):
        for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
            if (candidate / marker).exists():
                return candidate
        raise FileNotFoundError(f"Could not find project root (looking for {marker})")

    PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import pandas as pd

from src.models.common import data
from src.models.common.compare import find_cached_experiment, compare
from src.models.common.plotting import plot_loss_curve, plot_mae_curve, plot_prediction_curve
from src.models.transformer.training import run_training, load_cached_results, timestamp_now
from src.models.transformer.experiments.exp03_city_and_temperature_features.config import (
    CONFIG, EXPERIMENT_NAME, FEATURE_COLUMNS, CITY_EMBED_DIM
)

## Data

Same source as the baseline: `data/splits/{train,dev,test}.csv`, derived from `data/raw/*.csv` via `src/data/preprocess.py` -> `src/data/split_dataset.py` (chronological 70/15/15 split per city). `data.load_splits()` regenerates `data/splits/` automatically if missing.

In [ ]:
train_df, dev_df, test_df = data.load_splits()

print("Train rows:", len(train_df))
print("Dev rows  :", len(dev_df))
print("Test rows :", len(test_df))
train_df.head()

## Configuration

Hyperparameters (`CONFIG`) are identical to the baseline Transformer run. `FEATURE_COLUMNS` is exp01's list (baseline features plus the target's own recent history), and city identity is passed in as a second model input -- `data.CITY_VOCAB` maps each city name to an integer id, and `CITY_EMBED_DIM` sets the size of the learned embedding for it.


In [ ]:
print(CONFIG)
print()
print("Feature columns:", FEATURE_COLUMNS)
print("City embed dim: ", CITY_EMBED_DIM)
print("City vocab (", data.NUM_CITIES, "cities ):", data.CITY_VOCAB)

## Train or load

Same caching behavior as the baseline/exp01/exp02 notebooks: this experiment's runs live under their own `models/Transformer/exp03_city_and_temperature_features/` directory, so an exact `CONFIG` match here only ever compares against other exp03_city_and_temperature_features runs. Set `FORCE_RETRAIN = True` to bypass the cache and retrain regardless.


In [ ]:
FORCE_RETRAIN = False

MODEL_ROOT = data.PROJECT_ROOT / "models" / "Transformer" / EXPERIMENT_NAME
checkpoint_path = MODEL_ROOT / "checkpoints" / "best.keras"

config_dict = CONFIG.to_dict()
cached_dir = None if FORCE_RETRAIN else find_cached_experiment(f"Transformer/{EXPERIMENT_NAME}", config_dict)

if cached_dir:
    print(f"Found a cached experiment matching this config: {cached_dir.name}")
    results = load_cached_results(cached_dir)
else:
    print("No cached experiment matches this config (or FORCE_RETRAIN=True) -- training now...")
    experiment_dir = MODEL_ROOT / "experiments" / timestamp_now()
    results = run_training(
        CONFIG, experiment_dir, checkpoint_path,
        feature_columns=FEATURE_COLUMNS, city_aware=True, city_embed_dim=CITY_EMBED_DIM,
        show_plots=True
    )

print("Experiment dir:", results["experiment_dir"])

## Results

In [ ]:
print(f"Test Loss : {results['loss']:.4f}")
print(f"Test MAE  : {results['mae']:.4f}")
print(f"Test RMSE : {results['rmse']:.4f}")
print(f"Epochs trained: {results['epochs_trained']}")

figure_dir = results["experiment_dir"] / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)

plot_loss_curve(results["history"], figure_dir, show=True)
plot_mae_curve(results["history"], figure_dir, show=True)
plot_prediction_curve(results["y_test"], results["predictions"], figure_dir, show=True)

pd.read_csv(results["experiment_dir"] / "predictions.csv").head(10)

## Comparison to the baseline, exp01, and exp02

Reuses `src/models/common/compare.py`'s `compare()`, passing an explicit list of model names so it can look inside this experiment's namespaced results directory alongside the baseline, exp01 (temperature-as-feature), and exp02 (city embedding).


In [ ]:
compare(["Transformer/baseline", "Transformer/exp01_temperature_feature", "Transformer/exp02_city_embedding", f"Transformer/{EXPERIMENT_NAME}"])


## Save results

**If running on Colab**, run the cell below to zip this experiment's output folder together with this executed notebook (so the professor can see the real run, including these plots and outputs) and download it as one file. Unzip it directly into your local repo checkout, review with `git status`/`git diff`, then commit and push from your own machine.

**If running locally**, your results are already sitting in the repo at `results["experiment_dir"]` -- nothing more to do.

In [ ]:
try:
    import google.colab
    from google.colab import files

    zip_name = f"{EXPERIMENT_NAME}_results.zip"
    notebook_path = f"src/models/transformer/experiments/{EXPERIMENT_NAME}/experiment.ipynb"
    results_glob = f"models/Transformer/{EXPERIMENT_NAME}"

    run(f"zip -r {zip_name} {results_glob} {notebook_path}")
    files.download(zip_name)
except ImportError:
    print("Not running on Colab -- results are already in your local repo checkout.")